In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

### Initial Dataset (60 records)

In [0]:
data = [(1, 'Eva', 2450), (2, 'Charlie', 3097), (3, 'Ivy', 1953), (4, 'Bob', 3924), (5, 'Bob', 4746), (6, 'Frank', 3120), (7, 'Frank', 1431), (8, 'Frank', 801), (9, 'Jack', 4388), (10, 'Eva', 3268), (11, 'Ivy', 2794), (12, 'Jack', 4153), (13, 'Alice', 4410), (14, 'Ivy', 2865), (15, 'Frank', 2337), (16, 'Grace', 2575), (17, 'Charlie', 2446), (18, 'Bob', 2462), (19, 'Frank', 682), (20, 'Charlie', 819), (21, 'Eva', 4654), (22, 'Jack', 3449), (23, 'Ivy', 4630), (24, 'Eva', 3654), (25, 'Bob', 2662), (26, 'Frank', 741), (27, 'Ivy', 3688), (28, 'Jack', 4729), (29, 'Alice', 2930), (30, 'Charlie', 1085), (31, 'David', 4096), (32, 'Helen', 523), (33, 'Jack', 868), (34, 'Eva', 3094), (35, 'Charlie', 4403), (36, 'Bob', 1953), (37, 'Frank', 1996), (38, 'Alice', 1930), (39, 'Grace', 4408), (40, 'Alice', 4145), (41, 'David', 2478), (42, 'Charlie', 3403), (43, 'David', 2553), (44, 'Grace', 2456), (45, 'Charlie', 3523), (46, 'Frank', 2434), (47, 'Charlie', 2844), (48, 'Jack', 3830), (49, 'Frank', 1889), (50, 'Grace', 3858), (51, 'Eva', 4345), (52, 'Alice', 1584), (53, 'Charlie', 1359), (54, 'Bob', 3538), (55, 'Charlie', 983), (56, 'Frank', 696), (57, 'Alice', 675), (58, 'Alice', 3153), (59, 'Alice', 1802), (60, 'Eva', 4376)]


In [0]:
path = "/Volumes/workspace/default/time-travel-prac"

In [0]:
df = spark.createDataFrame(data, ["id","name","amount"])

### write as delta table

In [0]:
df.write.format("delta").mode("overwrite").save(path)

### loading(geting) data from path

In [0]:
delta_df = spark.read.format("delta").load(path)


In [0]:
print("Initial Data")
delta_df.show()

Initial Data
+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  1|    Eva|  2450|
|  2|Charlie|  3097|
|  3|    Ivy|  1953|
|  4|    Bob|  3924|
|  5|    Bob|  4746|
|  6|  Frank|  3120|
|  7|  Frank|  1431|
|  8|  Frank|   801|
|  9|   Jack|  4388|
| 10|    Eva|  3268|
| 11|    Ivy|  2794|
| 12|   Jack|  4153|
| 13|  Alice|  4410|
| 14|    Ivy|  2865|
| 15|  Frank|  2337|
| 16|  Grace|  2575|
| 17|Charlie|  2446|
| 18|    Bob|  2462|
| 19|  Frank|   682|
| 20|Charlie|   819|
+---+-------+------+
only showing top 20 rows


###  Incremental Dataset (for MERGE)

In [0]:
new_data = [
    (1, "Alice", 3000),   # update
    (2, "Bob", 4000),     # update
    (61, "NewCust1", 2000),  # insert
    (62, "NewCust2", 3500)   # insert
]

In [0]:
new_df = spark.createDataFrame(new_data, ["id","name","amount"])


### append to delta table

In [0]:
new_df.write.format("delta").mode("append").save(path)


In [0]:
print("After INSERT")
spark.read.format("delta").load(path).display()

After INSERT


id,name,amount
1,Eva,2450
2,Charlie,3097
3,Ivy,1953
4,Bob,3924
5,Bob,4746
6,Frank,3120
7,Frank,1431
8,Frank,801
9,Jack,4388
10,Eva,3268


### update existing records

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark,path)

# Update: increase amount by 500 where name = 'Eva'
delta_table.update(
    condition = "name = 'Eva'",
    set = {"amount": "amount + 500"}
)

print("After UPDATE")
delta_table.toDF().show()

After UPDATE
+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  2|Charlie|  3097|
|  3|    Ivy|  1953|
|  4|    Bob|  3924|
|  5|    Bob|  4746|
|  6|  Frank|  3120|
|  7|  Frank|  1431|
|  8|  Frank|   801|
|  9|   Jack|  4388|
| 11|    Ivy|  2794|
| 12|   Jack|  4153|
| 13|  Alice|  4410|
| 14|    Ivy|  2865|
| 15|  Frank|  2337|
| 16|  Grace|  2575|
| 17|Charlie|  2446|
| 18|    Bob|  2462|
| 19|  Frank|   682|
| 20|Charlie|   819|
| 22|   Jack|  3449|
| 23|    Ivy|  4630|
+---+-------+------+
only showing top 20 rows


In [0]:
delta_table.delete("amount < 1000")

print("After DELETE")
delta_table.toDF().show()

After DELETE
+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  1|    Eva|  2950|
| 10|    Eva|  3768|
| 21|    Eva|  5154|
| 24|    Eva|  4154|
| 34|    Eva|  3594|
| 51|    Eva|  4845|
| 60|    Eva|  4876|
|  2|Charlie|  3097|
|  3|    Ivy|  1953|
|  4|    Bob|  3924|
|  5|    Bob|  4746|
|  6|  Frank|  3120|
|  7|  Frank|  1431|
|  9|   Jack|  4388|
| 11|    Ivy|  2794|
| 12|   Jack|  4153|
| 13|  Alice|  4410|
| 14|    Ivy|  2865|
| 15|  Frank|  2337|
| 16|  Grace|  2575|
+---+-------+------+
only showing top 20 rows


### Merging the old data and new data with condition

In [0]:
delta_table.alias("target").merge(
    new_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set = {
    "name": "source.name",
    "amount": "source.amount"
}).whenNotMatchedInsert(values = {
    "id": "source.id",
    "name": "source.name",
    "amount": "source.amount"
}).execute()

print("After MERGE")
delta_table.toDF().show()

After MERGE
+---+-------+------+
| id|   name|amount|
+---+-------+------+
| 10|    Eva|  3768|
| 21|    Eva|  5154|
| 24|    Eva|  4154|
| 34|    Eva|  3594|
| 51|    Eva|  4845|
| 60|    Eva|  4876|
|  3|    Ivy|  1953|
|  4|    Bob|  3924|
|  5|    Bob|  4746|
|  6|  Frank|  3120|
|  7|  Frank|  1431|
|  9|   Jack|  4388|
| 11|    Ivy|  2794|
| 12|   Jack|  4153|
| 13|  Alice|  4410|
| 14|    Ivy|  2865|
| 15|  Frank|  2337|
| 16|  Grace|  2575|
| 17|Charlie|  2446|
| 18|    Bob|  2462|
+---+-------+------+
only showing top 20 rows


### schema evolution (when we want to add new column into existed delta table we use schema evolution)

In [0]:
# Add new column in incremental dataset
inc_new = [
    (65, "NewCust3", 2700, "Premium"),
    (66, "NewCust4", 1500, "Standard")
]

inc_new_df = spark.createDataFrame(inc_new, ["id","name","amount","category"])

# Enable schema evolution
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

delta_table.alias("target").merge(
    inc_new_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("After Schema Evolution")
delta_table.toDF().display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8803943785973652>, line 10
      7 inc_new_df = spark.createDataFrame(inc_new, ["id","name","amount","category"])
      9 # Enable schema evolution
---> 10 spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
     12 delta_table.alias("target").merge(
     13     inc_new_df.alias("source"),
     14     "target.id = source.id"
     15 ).whenMatchedUpdateAll() \
     16  .whenNotMatchedInsertAll() \
     17  .execute()
     19 print("After Schema Evolution")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:46, in RuntimeConf.set(self, key, value)
     44 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     45 operation = proto.ConfigRequest.Operation(set=op_set)
---> 46 result = self._client.config(operation)
     47 for warn in result.warni

### Time travel

In [0]:
spark.sql("DESCRIBE HISTORY delta.`/Volumes/workspace/default/time-travel-prac`").show(truncate=False)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, "/Volumes/workspace/default/time-travel-prac")

In [0]:
delta_table.history()

DataFrame[version: bigint, timestamp: timestamp, userId: string, userName: string, operation: string, operationParameters: map<string,string>, job: struct<jobId:string,jobName:string,jobRunId:string,runId:string,jobOwnerId:string,triggerType:string>, notebook: struct<notebookId:string>, queryHistoryStatementId: string, clusterId: string, readVersion: bigint, isolationLevel: string, isBlindAppend: boolean, operationMetrics: map<string,string>, userMetadata: string, engineInfo: string]